# Notebook 04 — Mathematical Portfolio Construction and Optimization

## Purpose
Construct and compare portfolios using SPY, EFA, EEM, AGG, GLD, XLE and XLK using development data only through 2021-12-31.

Methods:
1. Equal Weight
2. GMVP
3. Maximum Sharpe
4. Risk Parity
5. Minimum Historical Expected Shortfall (95%)

All optimized portfolios are long-only, fully invested, unlevered, and every asset included in a selected subset must have at least 5% weight.

Core principle: Mathematics generates candidate portfolios; the client mandate determines suitability.


In [1]:
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import minimize
from sklearn.covariance import LedoitWolf


In [2]:
PROJECT_ROOT = Path("..")

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"
FIGURE_DIR = PROJECT_ROOT / "outputs" / "figures"

SIMPLE_RETURN_FILE = PROCESSED_DIR / "simple_returns.csv"
RF_FILE = PROCESSED_DIR / "risk_free_daily_aligned.csv"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT.resolve())
print("Simple returns exists:", SIMPLE_RETURN_FILE.exists())
print("Risk-free file exists:", RF_FILE.exists())


Project root: C:\quant_edge_round2
Simple returns exists: True
Risk-free file exists: True


In [3]:
simple_returns = pd.read_csv(
    SIMPLE_RETURN_FILE,
    index_col=0,
    parse_dates=True
).sort_index()

TRAIN_END = "2021-12-31"

TICKERS = ["SPY", "EFA", "EEM", "AGG", "GLD", "XLE", "XLK"]

returns_train = (
    simple_returns
    .loc[:TRAIN_END, TICKERS]
    .copy()
)

print("Development period start:", returns_train.index.min())
print("Development period end:", returns_train.index.max())
print("Development observations:", len(returns_train))
print("Number of assets:", returns_train.shape[1])

assert returns_train.shape[1] == 7
assert returns_train.isna().sum().sum() == 0
assert returns_train.index.max() <= pd.Timestamp(TRAIN_END)


Development period start: 2004-11-19 00:00:00
Development period end: 2021-12-31 00:00:00
Development observations: 4309
Number of assets: 7


In [4]:
TRADING_DAYS = 252

expected_returns = returns_train.mean() * TRADING_DAYS
sample_covariance = returns_train.cov() * TRADING_DAYS

lw_model = LedoitWolf()
lw_model.fit(returns_train.values)

lw_covariance = pd.DataFrame(
    lw_model.covariance_ * TRADING_DAYS,
    index=TICKERS,
    columns=TICKERS
)

mu = expected_returns.copy()
cov_matrix = lw_covariance.copy()

covariance_comparison = pd.DataFrame({
    "Sample Volatility": np.sqrt(np.diag(sample_covariance)),
    "Ledoit-Wolf Volatility": np.sqrt(np.diag(lw_covariance))
}, index=TICKERS)

display(expected_returns)
display(covariance_comparison)


SPY    0.119341
EFA    0.077998
EEM    0.110702
AGG    0.038966
GLD    0.095099
XLE    0.099498
XLK    0.160539
dtype: float64

,Sample Volatility,Ledoit-Wolf Volatility
SPY,0.192376,0.192563
EFA,0.219176,0.219171
EEM,0.292800,0.292363
AGG,0.048709,0.051738
GLD,0.179955,0.180241
XLE,0.307007,0.306497
XLK,0.214523,0.214549


In [5]:
rf_data = pd.read_csv(
    RF_FILE,
    index_col=0,
    parse_dates=True
).sort_index()

if "Risk Free Daily" not in rf_data.columns:
    raise ValueError(
        "Expected column 'Risk Free Daily'."
    )

rf_daily = (
    rf_data["Risk Free Daily"]
    .reindex(
        returns_train.index
    )
)

assert rf_daily.isna().sum() == 0

annual_rf = (
    rf_daily.mean()
    * TRADING_DAYS
)

print(
    "Development-period risk-free rate "
    "(arithmetic annualized):",
    f"{annual_rf:.4%}"
)

Development-period risk-free rate (arithmetic annualized): 1.1775%


In [6]:
def portfolio_return(weights, expected_returns):
    return np.dot(weights, expected_returns)


def portfolio_volatility(weights, covariance_matrix):
    variance = weights @ covariance_matrix @ weights
    return np.sqrt(variance)


def portfolio_sharpe(weights, expected_returns, covariance_matrix, risk_free_rate):
    port_return = portfolio_return(weights, expected_returns)
    port_volatility = portfolio_volatility(weights, covariance_matrix)
    return (port_return - risk_free_rate) / port_volatility


def portfolio_historical_returns(weights, asset_returns):
    return asset_returns.values @ weights


def historical_var(portfolio_returns, confidence=0.95):
    cutoff = np.quantile(portfolio_returns, 1 - confidence)
    return -cutoff


def historical_es(portfolio_returns, confidence=0.95):
    cutoff = np.quantile(portfolio_returns, 1 - confidence)
    tail_returns = portfolio_returns[portfolio_returns <= cutoff]
    return -tail_returns.mean()


def risk_contributions(weights, covariance_matrix):
    portfolio_vol = portfolio_volatility(weights, covariance_matrix)
    marginal_risk = (covariance_matrix @ weights) / portfolio_vol
    return weights * marginal_risk


In [7]:
MIN_WEIGHT = 0.05

def make_portfolio_bounds(n, min_weight=MIN_WEIGHT):
    if n * min_weight > 1:
        raise ValueError("Minimum weight infeasible.")
    return [(min_weight, 1.0) for _ in range(n)]


def make_full_investment_constraint():
    return [{"type": "eq", "fun": lambda w: np.sum(w) - 1}]

print("Minimum allocation per selected asset:", f"{MIN_WEIGHT:.0%}")


Minimum allocation per selected asset: 5%


### Minimum Position Constraint

A 5% minimum allocation is imposed on every asset included in a candidate
subset.

This prevents the optimizer from including assets at economically negligible
weights simply to satisfy a larger asset-count classification.

The 5% floor is a modelling assumption rather than a client-specified policy
constraint and is therefore reported explicitly as a limitation.

In [8]:
def gmvp_objective(weights, covariance_matrix):
    return portfolio_volatility(weights, covariance_matrix)


def negative_sharpe(weights, expected_returns, covariance_matrix, risk_free_rate):
    return -portfolio_sharpe(
        weights,
        expected_returns,
        covariance_matrix,
        risk_free_rate
    )


def risk_parity_objective(weights, covariance_matrix):
    rc = risk_contributions(weights, covariance_matrix)
    target_rc = np.sum(rc) / len(weights)
    return np.sum((rc - target_rc) ** 2)


def minimum_es_objective(weights, subset_returns, confidence=0.95):
    port_returns = subset_returns.values @ weights
    return historical_es(port_returns, confidence)


In [9]:
def estimate_subset_inputs(subset_returns):
    expected = subset_returns.mean() * TRADING_DAYS

    lw = LedoitWolf()
    lw.fit(subset_returns.values)

    covariance = lw.covariance_ * TRADING_DAYS
    return expected, covariance


In [10]:
def optimize_gmvp(subset_returns):
    n = subset_returns.shape[1]
    _, covariance = estimate_subset_inputs(subset_returns)
    initial = np.repeat(1 / n, n)

    result = minimize(
        gmvp_objective,
        initial,
        args=(covariance,),
        method="SLSQP",
        bounds=make_portfolio_bounds(n),
        constraints=make_full_investment_constraint(),
        options={"maxiter": 1000, "ftol": 1e-12}
    )

    return result, covariance


In [11]:
def optimize_max_sharpe(subset_returns, risk_free_rate):
    n = subset_returns.shape[1]
    expected, covariance = estimate_subset_inputs(subset_returns)
    initial = np.repeat(1 / n, n)

    result = minimize(
        negative_sharpe,
        initial,
        args=(expected.values, covariance, risk_free_rate),
        method="SLSQP",
        bounds=make_portfolio_bounds(n),
        constraints=make_full_investment_constraint(),
        options={"maxiter": 1000, "ftol": 1e-12}
    )

    return result, expected, covariance


In [12]:
def optimize_risk_parity(subset_returns):
    n = subset_returns.shape[1]
    _, covariance = estimate_subset_inputs(subset_returns)
    initial = np.repeat(1 / n, n)

    result = minimize(
        risk_parity_objective,
        initial,
        args=(covariance,),
        method="SLSQP",
        bounds=make_portfolio_bounds(n),
        constraints=make_full_investment_constraint(),
        options={"maxiter": 1000, "ftol": 1e-12}
    )

    return result, covariance


In [13]:
def optimize_min_es_fast(subset_returns, confidence=0.95):
    n = subset_returns.shape[1]
    initial = np.repeat(1 / n, n)

    result = minimize(
        minimum_es_objective,
        initial,
        args=(subset_returns, confidence),
        method="SLSQP",
        bounds=make_portfolio_bounds(n),
        constraints=make_full_investment_constraint(),
        options={"maxiter": 1000, "ftol": 1e-10}
    )

    return result


### Minimum-ES implementation note
Historical ES is non-smooth. For the 120-subset screening stage, this notebook uses SLSQP directly on Historical ES because it is computationally practical. Exact Rockafellar-Uryasev LP validation can be reserved for shortlisted finalists.


In [14]:
def evaluate_portfolio(
    weights,
    subset_returns,
    expected_returns,
    covariance_matrix,
    risk_free_rate
):
    port_returns = portfolio_historical_returns(weights, subset_returns)

    return {
        "Expected Return": portfolio_return(weights, expected_returns),
        "Volatility": portfolio_volatility(weights, covariance_matrix),
        "Sharpe Ratio": portfolio_sharpe(
            weights,
            expected_returns,
            covariance_matrix,
            risk_free_rate
        ),
        "VaR 95%": historical_var(port_returns, 0.95),
        "ES 95%": historical_es(port_returns, 0.95),
        "VaR 99%": historical_var(port_returns, 0.99),
        "ES 99%": historical_es(port_returns, 0.99)
    }


## Full-universe portfolios

In [15]:
n_assets = len(TICKERS)
equal_weights = np.repeat(1 / n_assets, n_assets)

eq_metrics = evaluate_portfolio(
    equal_weights,
    returns_train,
    mu.values,
    cov_matrix.values,
    annual_rf
)

gmvp_result, gmvp_cov = optimize_gmvp(returns_train)
assert gmvp_result.success, gmvp_result.message
gmvp_weights = gmvp_result.x
gmvp_metrics = evaluate_portfolio(
    gmvp_weights,
    returns_train,
    mu.values,
    gmvp_cov,
    annual_rf
)

max_sharpe_result, max_sharpe_mu, max_sharpe_cov = optimize_max_sharpe(
    returns_train,
    annual_rf
)
assert max_sharpe_result.success, max_sharpe_result.message
max_sharpe_weights = max_sharpe_result.x
max_sharpe_metrics = evaluate_portfolio(
    max_sharpe_weights,
    returns_train,
    max_sharpe_mu.values,
    max_sharpe_cov,
    annual_rf
)

risk_parity_result, risk_parity_cov = optimize_risk_parity(returns_train)
assert risk_parity_result.success, risk_parity_result.message
risk_parity_weights = risk_parity_result.x
risk_parity_metrics = evaluate_portfolio(
    risk_parity_weights,
    returns_train,
    mu.values,
    risk_parity_cov,
    annual_rf
)

min_es_result = optimize_min_es_fast(returns_train, 0.95)
assert min_es_result.success, min_es_result.message
min_es_weights = min_es_result.x
min_es_metrics = evaluate_portfolio(
    min_es_weights,
    returns_train,
    mu.values,
    cov_matrix.values,
    annual_rf
)


C:\Users\Jani\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\scipy\optimize\_slsqp_py.py:435: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  fx = wrapped_fun(x)
C:\Users\Jani\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\scipy\optimize\_slsqp_py.py:439: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  g = append(wrapped_grad(x), 0.0)
C:\Users\Jani\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\scipy\optimize\_slsqp_py.py:493: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  a_eq = vstack([con['jac'](x, *con['args'])
C:\Users\Jani\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-package

In [16]:
full_universe_weights = {
    "Equal Weight": equal_weights,
    "GMVP": gmvp_weights,
    "Maximum Sharpe": max_sharpe_weights,
    "Risk Parity": risk_parity_weights,
    "Minimum ES": min_es_weights
}

full_universe_comparison = pd.DataFrame([
    {"Portfolio": "Equal Weight", **eq_metrics},
    {"Portfolio": "GMVP", **gmvp_metrics},
    {"Portfolio": "Maximum Sharpe", **max_sharpe_metrics},
    {"Portfolio": "Risk Parity", **risk_parity_metrics},
    {"Portfolio": "Minimum ES", **min_es_metrics}
]).set_index("Portfolio")

full_universe_weight_table = pd.DataFrame(
    full_universe_weights,
    index=TICKERS
)

display(full_universe_comparison)
display(full_universe_weight_table)


,Expected Return,Volatility,Sharpe Ratio,VaR 95%,ES 95%,VaR 99%,ES 99%
Portfolio,,,,,,,
Equal Weight,0.100306,0.164436,0.538392,0.014697,0.024765,0.030285,0.044691
GMVP,0.060435,0.067014,0.726113,0.005241,0.009171,0.010477,0.018022
Maximum Sharpe,0.080810,0.089692,0.769689,0.007937,0.013019,0.015368,0.023615
Risk Parity,0.069638,0.078200,0.739939,0.006648,0.011177,0.013321,0.020810
Minimum ES,0.060435,0.067014,0.726113,0.005241,0.009171,0.010477,0.018022


,Equal Weight,GMVP,Maximum Sharpe,Risk Parity,Minimum ES
SPY,0.142857,0.05,0.050000,0.071674,0.05
EFA,0.142857,0.05,0.050000,0.061065,0.05
EEM,0.142857,0.05,0.050000,0.050000,0.05
AGG,0.142857,0.70,0.465063,0.565086,0.70
GLD,0.142857,0.05,0.175114,0.132403,0.05
XLE,0.142857,0.05,0.050000,0.050000,0.05
XLK,0.142857,0.05,0.159823,0.069773,0.05


In [17]:
rp_rc = risk_contributions(
    risk_parity_weights,
    risk_parity_cov
)

rp_rc_percent = rp_rc / rp_rc.sum()

pd.DataFrame({
    "Weight": risk_parity_weights,
    "Risk Contribution %": rp_rc_percent
}, index=TICKERS)


,Weight,Risk Contribution %
SPY,0.071674,0.141009
EFA,0.061065,0.141194
EEM,0.050000,0.149747
AGG,0.565086,0.139105
GLD,0.132403,0.141790
XLE,0.050000,0.146024
XLK,0.069773,0.141131


## Constrained efficient frontier

In [18]:
def minimum_volatility_for_target_return(
    target_return,
    expected_returns,
    covariance_matrix
):
    n = len(expected_returns)
    initial = np.repeat(1 / n, n)

    constraints = [
        {"type": "eq", "fun": lambda w: np.sum(w) - 1},
        {
            "type": "eq",
            "fun": lambda w: np.dot(w, expected_returns) - target_return
        }
    ]

    return minimize(
        gmvp_objective,
        initial,
        args=(covariance_matrix,),
        method="SLSQP",
        bounds=make_portfolio_bounds(n),
        constraints=constraints,
        options={"maxiter": 1000, "ftol": 1e-10}
    )


In [19]:
base_weight = np.repeat(MIN_WEIGHT, len(TICKERS))
remaining_weight = 1 - base_weight.sum()

min_feasible_return = (
    np.dot(base_weight, mu.values)
    + remaining_weight * mu.min()
)

max_feasible_return = (
    np.dot(base_weight, mu.values)
    + remaining_weight * mu.max()
)

frontier_targets = np.linspace(
    min_feasible_return,
    max_feasible_return,
    40
)

frontier_rows = []

for target in frontier_targets:
    result = minimum_volatility_for_target_return(
        target,
        mu.values,
        cov_matrix.values
    )

    if result.success:
        frontier_rows.append({
            "Expected Return": target,
            "Volatility": portfolio_volatility(
                result.x,
                cov_matrix.values
            )
        })

efficient_frontier = pd.DataFrame(frontier_rows)
efficient_frontier.head()


,Expected Return,Volatility
0,0.060435,0.067014
1,0.062461,0.068485
2,0.064487,0.070461
3,0.066514,0.072583
4,0.068540,0.074795


In [20]:
plt.figure(figsize=(8, 6))

plt.plot(
    efficient_frontier["Volatility"],
    efficient_frontier["Expected Return"]
)

plt.scatter(
    full_universe_comparison["Volatility"],
    full_universe_comparison["Expected Return"]
)

for name, row in full_universe_comparison.iterrows():
    plt.annotate(
        name,
        (row["Volatility"], row["Expected Return"])
    )

plt.xlabel("Annualized Volatility")
plt.ylabel("Expected Annual Return")
plt.title("Constrained Efficient Frontier — Development Sample")
plt.grid(True, alpha=0.3)
plt.tight_layout()

plt.savefig(
    FIGURE_DIR / "efficient_frontier_development.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


C:\Users\Jani\AppData\Local\Temp\ipykernel_21192\3205292771.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Generate every 2–7 asset subset

In [21]:
all_subsets = []

for n in range(2, 8):
    for subset in combinations(TICKERS, n):
        all_subsets.append(subset)

print("Total subsets:", len(all_subsets))

subset_count = pd.Series(
    [len(subset) for subset in all_subsets]
).value_counts().sort_index()

subset_count


Total subsets: 120


2    21
3    35
4    35
5    21
6     7
7     1
Name: count, dtype: int64

In [22]:
optimization_failures = []


In [23]:
subset_results = []

total_subsets = len(all_subsets)

for subset_number, subset_tuple in enumerate(
    all_subsets,
    start=1
):

    print(
        f"Running subset "
        f"{subset_number}/{total_subsets}: "
        f"{subset_tuple}"
    )

    subset = list(subset_tuple)
    subset_returns = returns_train[subset]
    n = len(subset)

    expected, covariance = estimate_subset_inputs(
        subset_returns
    )

    # Equal Weight
    ew_weights = np.repeat(1 / n, n)

    ew_metrics = evaluate_portfolio(
        ew_weights,
        subset_returns,
        expected.values,
        covariance,
        annual_rf
    )

    subset_results.append({
        "N": n,
        "Subset": ", ".join(subset),
        "Method": "Equal Weight",
        "Weights": ew_weights,
        **ew_metrics
    })

    # GMVP
    gmvp_res, gmvp_cov = optimize_gmvp(
        subset_returns
    )

    if not gmvp_res.success:
        optimization_failures.append(
        {
            "Subset": ", ".join(subset),
            "Method": "GMVP",
            "Message": gmvp_res.message
        }
    )
    gmvp_w = gmvp_res.x

    gmvp_metrics = evaluate_portfolio(
        gmvp_w,
        subset_returns,
        expected.values,
        gmvp_cov,
        annual_rf
    )

    subset_results.append({
        "N": n,
        "Subset": ", ".join(subset),
        "Method": "GMVP",
        "Weights": gmvp_w,
        **gmvp_metrics
    })

    # Maximum Sharpe
    ms_res, ms_expected, ms_cov = optimize_max_sharpe(
        subset_returns,
        annual_rf
    )

    if not ms_res.success:
        optimization_failures.append(
        {
            "Subset": ", ".join(subset),
            "Method": "Maximum Sharpe",
            "Message": ms_res.message
        }
    )
    ms_w = ms_res.x

    ms_metrics = evaluate_portfolio(
        ms_w,
        subset_returns,
        ms_expected.values,
        ms_cov,
        annual_rf
    )

    subset_results.append({
        "N": n,
        "Subset": ", ".join(subset),
        "Method": "Maximum Sharpe",
        "Weights": ms_w,
        **ms_metrics
    })

    # Risk Parity
    rp_res, rp_cov = optimize_risk_parity(
        subset_returns
    )

    if not rp_res.success:
        optimization_failures.append(
        {
            "Subset": ", ".join(subset),
            "Method": "Risk Parity",
            "Message": rp_res.message
        }
    )
    rp_w = rp_res.x

    rp_metrics = evaluate_portfolio(
        rp_w,
        subset_returns,
        expected.values,
        rp_cov,
        annual_rf
    )

    subset_results.append({
        "N": n,
        "Subset": ", ".join(subset),
        "Method": "Risk Parity",
        "Weights": rp_w,
        **rp_metrics
    })

    # Minimum ES
    es_res = optimize_min_es_fast(
        subset_returns,
        confidence=0.95
    )

    if not es_res.success:
        optimization_failures.append(
        {
            "Subset": ", ".join(subset),
            "Method": "Minimum ES",
            "Message": es_res.message
        }
    )
    es_w = es_res.x

    es_metrics = evaluate_portfolio(
        es_w,
        subset_returns,
        expected.values,
        covariance,
        annual_rf
    )

    subset_results.append({
        "N": n,
        "Subset": ", ".join(subset),
        "Method": "Minimum ES",
        "Weights": es_w,
        **es_metrics
    })


C:\Users\Jani\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\scipy\optimize\_slsqp_py.py:435: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  fx = wrapped_fun(x)
C:\Users\Jani\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\scipy\optimize\_slsqp_py.py:439: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  g = append(wrapped_grad(x), 0.0)
C:\Users\Jani\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\scipy\optimize\_slsqp_py.py:493: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  a_eq = vstack([con['jac'](x, *con['args'])
C:\Users\Jani\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-package

Running subset 1/120: ('SPY', 'EFA')
Running subset 2/120: ('SPY', 'EEM')
Running subset 3/120: ('SPY', 'AGG')
Running subset 4/120: ('SPY', 'GLD')


Running subset 5/120: ('SPY', 'XLE')


C:\Users\Jani\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\scipy\optimize\_slsqp_py.py:435: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  fx = wrapped_fun(x)
C:\Users\Jani\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\scipy\optimize\_slsqp_py.py:435: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  fx = wrapped_fun(x)


C:\Users\Jani\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\scipy\optimize\_slsqp_py.py:439: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  g = append(wrapped_grad(x), 0.0)
C:\Users\Jani\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\scipy\optimize\_slsqp_py.py:493: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  a_eq = vstack([con['jac'](x, *con['args'])
C:\Users\Jani\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\scipy\optimize\_slsqp_py.py:435: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  fx = wrapped_fun(x)
C:\Users\Jani\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-package

Running subset 6/120: ('SPY', 'XLK')
Running subset 7/120: ('EFA', 'EEM')
Running subset 8/120: ('EFA', 'AGG')
Running subset 9/120: ('EFA', 'GLD')


... (69 outputs left out)

Running subset 117/120: ('SPY', 'EFA', 'AGG', 'GLD', 'XLE', 'XLK')
Running subset 118/120: ('SPY', 'EEM', 'AGG', 'GLD', 'XLE', 'XLK')
Running subset 119/120: ('EFA', 'EEM', 'AGG', 'GLD', 'XLE', 'XLK')


C:\Users\Jani\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\scipy\optimize\_slsqp_py.py:435: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  fx = wrapped_fun(x)
C:\Users\Jani\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\scipy\optimize\_slsqp_py.py:435: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  fx = wrapped_fun(x)
C:\Users\Jani\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\scipy\optimize\_slsqp_py.py:435: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  fx = wrapped_fun(x)
C:\Users\Jani\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\scipy\opti

Running subset 120/120: ('SPY', 'EFA', 'EEM', 'AGG', 'GLD', 'XLE', 'XLK')


C:\Users\Jani\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\scipy\optimize\_slsqp_py.py:435: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  fx = wrapped_fun(x)
C:\Users\Jani\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\scipy\optimize\_slsqp_py.py:439: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  g = append(wrapped_grad(x), 0.0)
C:\Users\Jani\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\scipy\optimize\_slsqp_py.py:493: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  a_eq = vstack([con['jac'](x, *con['args'])
C:\Users\Jani\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-package

In [24]:
print(
    "Optimization failures:",
    len(
        optimization_failures
    )
)

if optimization_failures:

    display(
        pd.DataFrame(
            optimization_failures
        )
    )

Optimization failures: 0


### SLSQP Numerical Warning

During some optimization steps, SciPy's SLSQP routine temporarily proposed
trial values outside the feasible bounds and internally clipped them back to
the permitted range.

These messages do not indicate failed optimization.

All retained solutions are subsequently checked for:

- successful optimizer convergence;
- fully invested weights;
- long-only allocations;
- minimum-position constraints; and
- correct asset counts.

No failed optimization was retained in the final candidate set.

## Validate asset-count rules

In [25]:
subset_results_df = pd.DataFrame(
    subset_results
)

print(
    "Total portfolio candidates:",
    len(
        subset_results_df
    )
)

subset_results_df.head()

Total portfolio candidates: 600


,N,Subset,Method,Weights,Expected Return,Volatility,Sharpe Ratio,VaR 95%,ES 95%,VaR 99%,ES 99%
0,2,"SPY, EFA",Equal Weight,"[0.5, 0.5]",0.098670,0.200010,0.434453,0.018389,0.030999,0.038302,0.056219
1,2,"SPY, EFA",GMVP,"[0.95, 0.05000000000000002]",0.117274,0.192672,0.547558,0.017757,0.029940,0.036317,0.053483
2,2,"SPY, EFA",Maximum Sharpe,"[0.95, 0.04999999999999999]",0.117274,0.192672,0.547558,0.017757,0.029940,0.036317,0.053483
3,2,"SPY, EFA",Risk Parity,"[0.5322426734787872, 0.4677573265212129]",0.100003,0.199154,0.443013,0.018272,0.030873,0.038458,0.055980
4,2,"SPY, EFA",Minimum ES,"[0.9500000000000001, 0.04999999999999999]",0.117274,0.192672,0.547558,0.017757,0.029940,0.036317,0.053483


In [26]:
def count_active_assets(weights, threshold=0.001):
    return np.sum(np.asarray(weights) > threshold)


subset_results_df["Active Assets"] = (
    subset_results_df["Weights"]
    .apply(count_active_assets)
)

subset_results_df["Correct Asset Count"] = (
    subset_results_df["Active Assets"]
    ==
    subset_results_df["N"]
)

print(
    "Candidates with fewer than 2 active assets:",
    (subset_results_df["Active Assets"] < 2).sum()
)

print(
    subset_results_df["Correct Asset Count"].value_counts()
)

assert len(subset_results_df) == 600
assert (subset_results_df["Active Assets"] >= 2).all()
assert subset_results_df["Correct Asset Count"].all()


Candidates with fewer than 2 active assets:

 0
Correct Asset Count
True    600
Name: count, dtype: int64


## Objective-based shortlists

In [27]:
best_equal_weight = (
    subset_results_df[
        subset_results_df["Method"] == "Equal Weight"
    ]
    .sort_values("Sharpe Ratio", ascending=False)
    .head(1)
)

best_gmvp = (
    subset_results_df[
        subset_results_df["Method"] == "GMVP"
    ]
    .sort_values("Volatility")
    .head(1)
)

best_max_sharpe = (
    subset_results_df[
        subset_results_df["Method"] == "Maximum Sharpe"
    ]
    .sort_values("Sharpe Ratio", ascending=False)
    .head(1)
)

best_min_es = (
    subset_results_df[
        subset_results_df["Method"] == "Minimum ES"
    ]
    .sort_values("ES 95%")
    .head(1)
)

objective_shortlist = pd.concat(
    [
        best_equal_weight,
        best_gmvp,
        best_max_sharpe,
        best_min_es
    ],
    ignore_index=True
)

objective_shortlist[
    [
        "N",
        "Subset",
        "Method",
        "Expected Return",
        "Volatility",
        "Sharpe Ratio",
        "VaR 95%",
        "ES 95%",
        "VaR 99%",
        "ES 99%"
    ]
]


,N,Subset,Method,Expected Return,Volatility,Sharpe Ratio,VaR 95%,ES 95%,VaR 99%,ES 99%
0,3,"AGG, GLD, XLK",Equal Weight,0.098201,0.096768,0.893125,0.008894,0.014034,0.015985,0.023854
1,2,"SPY, AGG",GMVP,0.045295,0.048037,0.697785,0.003446,0.006114,0.006490,0.012690
2,3,"AGG, GLD, XLK",Maximum Sharpe,0.072782,0.062788,0.971625,0.005363,0.008575,0.009713,0.015404
3,2,"SPY, AGG",Minimum ES,0.044859,0.048050,0.688525,0.003450,0.006110,0.006474,0.012667


### Risk Parity Candidate Selection

Risk Parity portfolios are first defined by the equal-risk-contribution
objective.

When multiple asset subsets satisfy this objective, performance metrics such
as Sharpe ratio may be used only as a secondary criterion for selecting which
Risk Parity subset proceeds to later validation.

The Sharpe ratio does not define the Risk Parity optimization itself.

### Minimum-ES validation note

Historical Expected Shortfall is a non-smooth objective, so SLSQP is used
during the large candidate-screening stage for computational simplicity.

The shortlisted Minimum-ES portfolio is retained only if the optimizer reports
successful convergence and the resulting portfolio subsequently performs
credibly under independent VaR, ES, EVT, out-of-sample and stress-testing
analysis.

The Minimum-ES result is therefore treated as a practical screening solution
rather than as proof of a global optimum.

## Best portfolio by number of assets

In [28]:
best_gmvp_by_n = (
    subset_results_df[
        subset_results_df["Method"] == "GMVP"
    ]
    .sort_values(["N", "Volatility"])
    .groupby("N", as_index=False)
    .first()
)

best_sharpe_by_n = (
    subset_results_df[
        subset_results_df["Method"] == "Maximum Sharpe"
    ]
    .sort_values(
        ["N", "Sharpe Ratio"],
        ascending=[True, False]
    )
    .groupby("N", as_index=False)
    .first()
)

best_es_by_n = (
    subset_results_df[
        subset_results_df["Method"] == "Minimum ES"
    ]
    .sort_values(["N", "ES 95%"])
    .groupby("N", as_index=False)
    .first()
)

display(best_gmvp_by_n[
    ["N", "Subset", "Volatility", "Expected Return", "Sharpe Ratio", "ES 95%"]
])

display(best_sharpe_by_n[
    ["N", "Subset", "Sharpe Ratio", "Expected Return", "Volatility", "ES 95%"]
])

display(best_es_by_n[
    ["N", "Subset", "ES 95%", "VaR 95%", "Expected Return", "Volatility", "Sharpe Ratio"]
])


,N,Subset,Volatility,Expected Return,Sharpe Ratio,ES 95%
0,2,"SPY, AGG",0.048037,0.045295,0.697785,0.006114
1,3,"SPY, AGG, XLK",0.048204,0.049063,0.773556,0.006194
2,4,"SPY, AGG, GLD, XLK",0.048638,0.051870,0.824351,0.006247
3,5,"SPY, EFA, AGG, GLD, XLK",0.051453,0.053822,0.817176,0.006620
4,6,"SPY, EFA, AGG, GLD, XLE, XLK",0.058205,0.056848,0.774383,0.007701
5,7,"SPY, EFA, EEM, AGG, GLD, XLE, XLK",0.067014,0.060435,0.726113,0.009171


,N,Subset,Sharpe Ratio,Expected Return,Volatility,ES 95%
0,2,"AGG, XLK",0.913342,0.066530,0.059950,0.008184
1,3,"AGG, GLD, XLK",0.971625,0.072782,0.062788,0.008575
2,4,"SPY, AGG, GLD, XLK",0.960588,0.071491,0.062166,0.008512
3,5,"SPY, EFA, AGG, GLD, XLK",0.898553,0.071432,0.066393,0.009218
4,6,"SPY, EFA, AGG, GLD, XLE, XLK",0.830098,0.075043,0.076217,0.010861
5,7,"SPY, EFA, EEM, AGG, GLD, XLE, XLK",0.769689,0.080810,0.089692,0.013019


,N,Subset,ES 95%,VaR 95%,Expected Return,Volatility,Sharpe Ratio
0,2,"SPY, AGG",0.006110,0.003450,0.044859,0.048050,0.688525
1,3,"SPY, AGG, GLD",0.006193,0.003516,0.047355,0.049125,0.724283
2,4,"SPY, AGG, GLD, XLK",0.006247,0.003548,0.051870,0.048638,0.824351
3,5,"SPY, EFA, AGG, GLD, XLK",0.006620,0.003777,0.053822,0.051453,0.817176
4,6,"SPY, EFA, AGG, GLD, XLE, XLK",0.007701,0.004398,0.056848,0.058205,0.774383
5,7,"SPY, EFA, EEM, AGG, GLD, XLE, XLK",0.009171,0.005241,0.060435,0.067014,0.726113


## Diversification summary

In [29]:
diversification_summary = pd.DataFrame({
    "N": range(2, 8)
})

min_vol_by_n = (
    subset_results_df[
        subset_results_df["Method"] == "GMVP"
    ]
    .groupby("N")["Volatility"]
    .min()
)

max_sharpe_by_n = (
    subset_results_df[
        subset_results_df["Method"] == "Maximum Sharpe"
    ]
    .groupby("N")["Sharpe Ratio"]
    .max()
)

min_es_by_n = (
    subset_results_df[
        subset_results_df["Method"] == "Minimum ES"
    ]
    .groupby("N")["ES 95%"]
    .min()
)

diversification_summary["Minimum Volatility"] = (
    diversification_summary["N"].map(min_vol_by_n)
)

diversification_summary["Maximum Sharpe"] = (
    diversification_summary["N"].map(max_sharpe_by_n)
)

diversification_summary["Minimum ES 95%"] = (
    diversification_summary["N"].map(min_es_by_n)
)

diversification_summary


,N,Minimum Volatility,Maximum Sharpe,Minimum ES 95%
0,2,0.048037,0.913342,0.006110
1,3,0.048204,0.971625,0.006193
2,4,0.048638,0.960588,0.006247
3,5,0.051453,0.898553,0.006620
4,6,0.058205,0.830098,0.007701
5,7,0.067014,0.769689,0.009171


## Diversification Result

Diversification benefits depend on asset quality and dependence structure,
not simply on the number of holdings.

Under the 5% minimum-position constraint, the best three-asset portfolio
achieves a higher Sharpe ratio than the full seven-asset portfolio.

Similarly, the minimum-volatility solution occurs in a smaller subset rather
than in the full universe.

This shows that adding assets can reduce portfolio efficiency when additional
holdings are highly correlated or offer weak marginal diversification
benefits.

In [30]:
plt.figure(figsize=(8, 5))

plt.plot(
    diversification_summary["N"],
    diversification_summary["Minimum Volatility"],
    marker="o"
)

plt.xlabel("Number of Assets")
plt.ylabel("Minimum Annualized Volatility")
plt.title("Best Minimum Volatility by Portfolio Size")
plt.grid(True, alpha=0.3)
plt.tight_layout()

plt.savefig(
    FIGURE_DIR / "diversification_min_volatility.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


C:\Users\Jani\AppData\Local\Temp\ipykernel_21192\2567015048.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [31]:
plt.figure(figsize=(8, 5))

plt.plot(
    diversification_summary["N"],
    diversification_summary["Minimum ES 95%"],
    marker="o"
)

plt.xlabel("Number of Assets")
plt.ylabel("Minimum Historical ES 95%")
plt.title("Best Minimum ES (95%) by Portfolio Size")
plt.grid(True, alpha=0.3)
plt.tight_layout()

plt.savefig(
    FIGURE_DIR / "diversification_min_es95.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


C:\Users\Jani\AppData\Local\Temp\ipykernel_21192\3759841903.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Candidate risk-return map

In [32]:
plt.figure(figsize=(9, 6))

for method in subset_results_df["Method"].unique():

    method_data = subset_results_df[
        subset_results_df["Method"] == method
    ]

    plt.scatter(
        method_data["Volatility"],
        method_data["Expected Return"],
        label=method,
        alpha=0.6
    )

plt.xlabel("Annualized Volatility")
plt.ylabel("Expected Annual Return")
plt.title("Development-Sample Candidate Portfolios")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

plt.savefig(
    FIGURE_DIR / "candidate_portfolios_risk_return.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


C:\Users\Jani\AppData\Local\Temp\ipykernel_21192\2574915609.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Save outputs

In [33]:
def weights_to_string(weights):
    return ";".join(
        f"{x:.10f}"
        for x in np.asarray(weights)
    )


subset_results_export = subset_results_df.copy()

subset_results_export["Weights"] = (
    subset_results_export["Weights"]
    .apply(weights_to_string)
)

subset_results_export.to_csv(
    TABLE_DIR / "portfolio_candidates_600_development.csv",
    index=False
)

full_universe_comparison.to_csv(
    TABLE_DIR / "full_universe_method_comparison.csv"
)

full_universe_weight_table.to_csv(
    TABLE_DIR / "full_universe_method_weights.csv"
)

efficient_frontier.to_csv(
    TABLE_DIR / "efficient_frontier_development.csv",
    index=False
)

diversification_summary.to_csv(
    TABLE_DIR / "diversification_summary_by_asset_count.csv",
    index=False
)

best_gmvp_by_n.to_csv(
    TABLE_DIR / "best_gmvp_by_asset_count.csv",
    index=False
)

best_sharpe_by_n.to_csv(
    TABLE_DIR / "best_max_sharpe_by_asset_count.csv",
    index=False
)

best_es_by_n.to_csv(
    TABLE_DIR / "best_min_es_by_asset_count.csv",
    index=False
)

print("Notebook 04 outputs saved successfully.")


Notebook 04 outputs saved successfully.


## Interpretation

Do not choose the final endowment portfolio from one metric.

These candidates must later be evaluated with:
- Sortino, Treynor, Jensen Alpha, Information Ratio and Calmar
- Parametric / Historical / Monte Carlo VaR
- Expected Shortfall
- EVT
- maximum drawdown
- risk contributions
- stress testing
- out-of-sample 2022+ validation
- VaR backtesting

A good investment decision is not the one with the highest expected return — it is the one whose downside has been measured, tested and can be defended.


In [34]:
assert returns_train.shape == (
    4309,
    7
)

assert returns_train.index.max() == pd.Timestamp(
    "2021-12-31"
)

assert len(
    subset_results_df
) == 600

assert (
    subset_results_df[
        "Correct Asset Count"
    ].all()
)

assert (
    subset_results_df[
        "Weights"
    ].apply(
        lambda w:
        np.isclose(
            np.sum(w),
            1.0,
            atol=1e-6
        )
    )
    .all()
)

assert (
    subset_results_df[
        "Weights"
    ].apply(
        lambda w:
        (
            np.asarray(w)
            >= MIN_WEIGHT - 1e-6
        ).all()
    )
    .all()
)

assert len(
    optimization_failures
) == 0

print(
    "All Notebook 04 final "
    "validation checks passed."
)

All Notebook 04 final validation checks passed.
